# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# 2. Define Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "fmcg_sales_marketing_profitability_2023_2025.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Processed data dir:", PROCESSED_DATA_DIR)

Project root: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new
Raw data path: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\raw\fmcg_sales_marketing_profitability_2023_2025.csv
Processed data dir: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed


# 3. Load Dataset

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

Rows: 18,240
Columns: 27


,Order_ID,Order_Date,Year,Quarter,Month,Month_Name,Region,Country,City,Sales_Person,Customer_Type,Sales_Channel,Promotion_Type,Product_Category,Brand,Product_Name,SKU,Units_Sold,Unit_Price_USD,Discount_Pct,Gross_Sales_USD,Marketing_Spend_USD,COGS_USD,Logistics_Cost_USD,Net_Revenue_USD,Profit_USD,Profit_Margin_Pct
0,FMCG-2025-000001,2025-09-26,2025,Q3,9,September,Oceania,Australia,Perth,Ethan Cole,B2C,Modern Trade,Seasonal Campaign,Personal Care,PureLiva,Shampoo Repair,PCR-PL-101,73,8.47,8.94,618.31,66.05,314.09,43.03,563.03,139.86,24.84
1,FMCG-2024-000002,2024-10-09,2024,Q4,10,October,Asia,India,Mumbai,Meera Nair,B2C,Online,Bundle Offer,Personal Care,BrightSmile,Toothpaste Mint,PCR-BS-301,99,2.89,9.86,286.11,35.26,123.79,23.89,257.90,74.96,29.07
2,FMCG-2024-000003,2024-07-06,2024,Q3,7,July,North America,USA,Los Angeles,Nina Booker,B2B,Distributor,Seasonal Campaign,Personal Care,FreshNest,Body Wash Citrus,PCR-FN-201,361,5.96,15.32,"2,151.56",171.46,"1,011.60",107.02,"1,821.94",531.86,29.19
3,FMCG-2024-000004,2024-05-25,2024,Q2,5,May,Europe,France,Paris,Oliver Kent,B2B,Wholesale,No Promo,Snacks,NutriBite,Protein Bar Peanut,SNK-NB-202,603,3.80,18.00,"2,291.40",118.39,"1,133.66",85.20,"1,878.95",541.70,28.83
4,FMCG-2023-000005,2023-08-10,2023,Q3,8,August,Europe,France,Lyon,Lucas Bennett,B2C,Modern Trade,Loyalty Cashback,Dairy & Breakfast,FarmJoy,Greek Yogurt Plain,DBF-FJ-202,113,3.18,12.46,359.34,34.82,157.70,23.02,314.57,99.03,31.48


# 4. Basic Cleaning and Date Parsing

In [4]:
df = df.copy()

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

# Standardize column names
string_cols = df.select_dtypes(include="object").columns.tolist()

for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# Recreate date-derived columns from Order_Date for consistency
df["Year"] = df["Order_Date"].dt.year
df["Quarter"] = "Q" + df["Order_Date"].dt.quarter.astype(str)
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Year_Month"] = df["Order_Date"].dt.to_period("M").astype(str)

# Sort data
df = df.sort_values("Order_Date").reset_index(drop=True)

df[["Order_Date", "Year", "Quarter", "Month", "Month_Name", "Year_Month"]].head()

,Order_Date,Year,Quarter,Month,Month_Name,Year_Month
0,2023-01-01,2023,Q1,1,January,2023-01
1,2023-01-01,2023,Q1,1,January,2023-01
2,2023-01-01,2023,Q1,1,January,2023-01
3,2023-01-01,2023,Q1,1,January,2023-01
4,2023-01-01,2023,Q1,1,January,2023-01


In [6]:
# Validate required columns

required_cols = [
    "Order_ID", "Order_Date", "Year", "Quarter", "Month", "Month_Name",
    "Region", "Country", "City",
    "Sales_Person", "Customer_Type", "Sales_Channel", "Promotion_Type",
    "Product_Category", "Brand", "Product_Name", "SKU",
    "Units_Sold", "Unit_Price_USD", "Discount_Pct", "Gross_Sales_USD",
    "Marketing_Spend_USD", "COGS_USD", "Logistics_Cost_USD",
    "Net_Revenue_USD", "Profit_USD", "Profit_Margin_Pct"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("All required columns are available.")

All required columns are available.


In [7]:
# Numeric conversion and safety checks

numeric_cols = [
    "Units_Sold",
    "Unit_Price_USD",
    "Discount_Pct",
    "Gross_Sales_USD",
    "Marketing_Spend_USD",
    "COGS_USD",
    "Logistics_Cost_USD",
    "Net_Revenue_USD",
    "Profit_USD",
    "Profit_Margin_Pct",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_quality = pd.DataFrame({
    "column": numeric_cols,
    "missing_count": [df[col].isna().sum() for col in numeric_cols],
    "negative_count": [(df[col] < 0).sum() for col in numeric_cols],
    "zero_count": [(df[col] == 0).sum() for col in numeric_cols],
    "min": [df[col].min() for col in numeric_cols],
    "max": [df[col].max() for col in numeric_cols],
})

numeric_quality

,column,missing_count,negative_count,zero_count,min,max
0,Units_Sold,0,0,0,5.00,"1,366.00"
1,Unit_Price_USD,0,0,0,1.45,12.74
2,Discount_Pct,0,0,1,0.00,28.00
3,Gross_Sales_USD,0,0,0,13.02,"10,634.12"
4,Marketing_Spend_USD,0,0,0,1.81,"1,767.71"
5,COGS_USD,0,0,0,6.31,"6,473.33"
6,Logistics_Cost_USD,0,0,0,1.08,628.86
7,Net_Revenue_USD,0,0,0,10.54,"8,752.94"
8,Profit_USD,0,784,2,-637.50,"2,723.00"
9,Profit_Margin_Pct,0,784,3,-45.31,44.49


In [8]:
# Handle invalid values conservatively

df["is_invalid_date"] = df["Order_Date"].isna().astype(int)
df["is_non_positive_units"] = (df["Units_Sold"] <= 0).astype(int)
df["is_non_positive_net_revenue"] = (df["Net_Revenue_USD"] <= 0).astype(int)
df["is_negative_profit"] = (df["Profit_USD"] < 0).astype(int)

# For ratio calculations only
df["safe_units_sold"] = df["Units_Sold"].replace(0, np.nan)
df["safe_net_revenue"] = df["Net_Revenue_USD"].replace(0, np.nan)
df["safe_marketing_spend"] = df["Marketing_Spend_USD"].replace(0, np.nan)

df[[
    "is_invalid_date",
    "is_non_positive_units",
    "is_non_positive_net_revenue",
    "is_negative_profit"
]].sum()

is_invalid_date                  0
is_non_positive_units            0
is_non_positive_net_revenue      0
is_negative_profit             784
dtype: int64

# 5. Business KPI Features

In [9]:
# Pricing and commercial metrics
df["Avg_Selling_Price_USD"] = df["Net_Revenue_USD"] / df["safe_units_sold"]
df["Gross_Sales_per_Unit_USD"] = df["Gross_Sales_USD"] / df["safe_units_sold"]
df["Discount_Amount_USD"] = df["Gross_Sales_USD"] - df["Net_Revenue_USD"]
df["Discount_Amount_USD"] = df["Discount_Amount_USD"].clip(lower=0)

# Cost and profitability ratios
df["COGS_Ratio"] = df["COGS_USD"] / df["safe_net_revenue"]
df["Logistics_Cost_Ratio"] = df["Logistics_Cost_USD"] / df["safe_net_revenue"]
df["Marketing_Spend_Ratio"] = df["Marketing_Spend_USD"] / df["safe_net_revenue"]
df["Profit_Margin"] = df["Profit_USD"] / df["safe_net_revenue"]

# Gross profit approximation
df["Gross_Profit_USD"] = df["Net_Revenue_USD"] - df["COGS_USD"]
df["Gross_Margin"] = df["Gross_Profit_USD"] / df["safe_net_revenue"]

# Contribution profit approximation after direct commercial costs
df["Contribution_Profit_USD"] = (
    df["Net_Revenue_USD"]
    - df["COGS_USD"]
    - df["Logistics_Cost_USD"]
    - df["Marketing_Spend_USD"]
)
df["Contribution_Margin"] = df["Contribution_Profit_USD"] / df["safe_net_revenue"]

# Marketing efficiency proxy
df["Marketing_Efficiency_Profit_per_USD"] = df["Profit_USD"] / df["safe_marketing_spend"]
df["Revenue_per_Marketing_USD"] = df["Net_Revenue_USD"] / df["safe_marketing_spend"]

# Cost per unit
df["COGS_per_Unit_USD"] = df["COGS_USD"] / df["safe_units_sold"]
df["Logistics_Cost_per_Unit_USD"] = df["Logistics_Cost_USD"] / df["safe_units_sold"]
df["Marketing_Spend_per_Unit_USD"] = df["Marketing_Spend_USD"] / df["safe_units_sold"]
df["Profit_per_Unit_USD"] = df["Profit_USD"] / df["safe_units_sold"]

df.head()

,Order_ID,Order_Date,Year,Quarter,Month,Month_Name,Region,Country,City,Sales_Person,Customer_Type,Sales_Channel,Promotion_Type,Product_Category,Brand,Product_Name,SKU,Units_Sold,Unit_Price_USD,Discount_Pct,Gross_Sales_USD,Marketing_Spend_USD,COGS_USD,Logistics_Cost_USD,Net_Revenue_USD,Profit_USD,Profit_Margin_Pct,Year_Month,is_invalid_date,is_non_positive_units,is_non_positive_net_revenue,is_negative_profit,safe_units_sold,safe_net_revenue,safe_marketing_spend,Avg_Selling_Price_USD,Gross_Sales_per_Unit_USD,Discount_Amount_USD,COGS_Ratio,Logistics_Cost_Ratio,Marketing_Spend_Ratio,Profit_Margin,Gross_Profit_USD,Gross_Margin,Contribution_Profit_USD,Contribution_Margin,Marketing_Efficiency_Profit_per_USD,Revenue_per_Marketing_USD,COGS_per_Unit_USD,Logistics_Cost_per_Unit_USD,Marketing_Spend_per_Unit_USD,Profit_per_Unit_USD
0,FMCG-2023-005244,2023-01-01,2023,Q1,1,January,Europe,UK,London,Lucas Bennett,B2B,Modern Trade,Seasonal Campaign,Household,SparkShield,Surface Cleaner,HHD-SS-201,89,5.97,19.73,531.33,59.40,215.18,36.45,426.50,115.47,27.07,2023-01,0,0,0,0,89,426.50,59.40,4.79,5.97,104.83,0.50,0.09,0.14,0.27,211.32,0.50,115.47,0.27,1.94,7.18,2.42,0.41,0.67,1.30
1,FMCG-2023-017472,2023-01-01,2023,Q1,1,January,Asia,Thailand,Bangkok,Diya Kapoor,B2C,Modern Trade,No Promo,Personal Care,FreshNest,Hand Soap Gentle,PCR-FN-202,22,2.69,2.77,59.18,6.14,30.88,3.63,57.54,16.89,29.35,2023-01,0,0,0,0,22,57.54,6.14,2.62,2.69,1.64,0.54,0.06,0.11,0.29,26.66,0.46,16.89,0.29,2.75,9.37,1.40,0.17,0.28,0.77
2,FMCG-2023-006588,2023-01-01,2023,Q1,1,January,Europe,Spain,Valencia,Sofia Mercer,B2B,Modern Trade,Seasonal Campaign,Dairy & Breakfast,PantryPeak,Peanut Butter Creamy,DBF-PP-301,294,6.27,18.17,"1,843.38",158.92,882.56,100.48,"1,508.44",366.48,24.30,2023-01,0,0,0,0,294,"1,508.44",158.92,5.13,6.27,334.94,0.59,0.07,0.11,0.24,625.88,0.41,366.48,0.24,2.31,9.49,3.00,0.34,0.54,1.25
3,FMCG-2023-011561,2023-01-01,2023,Q1,1,January,Europe,Germany,Berlin,Hugo Martin,B2B,Modern Trade,Flash Discount,Snacks,NutriBite,Protein Bar Cocoa,SNK-NB-201,260,3.67,21.63,954.20,128.43,431.12,49.75,747.81,138.51,18.52,2023-01,0,0,0,0,260,747.81,128.43,2.88,3.67,206.39,0.58,0.07,0.17,0.19,316.69,0.42,138.51,0.19,1.08,5.82,1.66,0.19,0.49,0.53
4,FMCG-2023-002026,2023-01-01,2023,Q1,1,January,Europe,UK,Manchester,Amelia Rhodes,B2B,Distributor,Bundle Offer,Personal Care,PureLiva,Shampoo Repair,PCR-PL-101,596,7.10,16.75,"4,231.60",331.27,"1,953.78",208.02,"3,522.81","1,029.74",29.23,2023-01,0,0,0,0,596,"3,522.81",331.27,5.91,7.10,708.79,0.55,0.06,0.09,0.29,"1,569.03",0.45,"1,029.74",0.29,3.11,10.63,3.28,0.35,0.56,1.73


In [10]:
# Check KPI reasonableness

kpi_cols = [
    "Avg_Selling_Price_USD",
    "Gross_Sales_per_Unit_USD",
    "Discount_Amount_USD",
    "COGS_Ratio",
    "Logistics_Cost_Ratio",
    "Marketing_Spend_Ratio",
    "Profit_Margin",
    "Gross_Margin",
    "Contribution_Margin",
    "Marketing_Efficiency_Profit_per_USD",
    "Revenue_per_Marketing_USD",
    "COGS_per_Unit_USD",
    "Logistics_Cost_per_Unit_USD",
    "Marketing_Spend_per_Unit_USD",
    "Profit_per_Unit_USD",
]

df[kpi_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Avg_Selling_Price_USD,"18,240.00",3.89,1.73,1.11,2.62,3.47,4.77,12.21
Gross_Sales_per_Unit_USD,"18,240.00",4.47,1.95,1.45,3.02,3.99,5.44,12.74
Discount_Amount_USD,"18,240.00",144.22,199.87,0.00,24.13,67.77,183.36,"2,540.19"
COGS_Ratio,"18,240.00",0.59,0.07,0.48,0.54,0.58,0.62,0.83
Logistics_Cost_Ratio,"18,240.00",0.08,0.02,0.04,0.06,0.08,0.10,0.14
Marketing_Spend_Ratio,"18,240.00",0.14,0.07,0.03,0.09,0.13,0.16,0.61
Profit_Margin,"18,240.00",0.20,0.11,-0.45,0.14,0.21,0.27,0.44
Gross_Margin,"18,240.00",0.41,0.07,0.17,0.38,0.42,0.46,0.52
Contribution_Margin,"18,240.00",0.20,0.11,-0.45,0.14,0.21,0.27,0.44
Marketing_Efficiency_Profit_per_USD,"18,240.00",2.15,1.95,-0.79,0.87,1.64,2.83,13.30


# 6. Monthly KPI Table

In [11]:
monthly_kpi = (
    df.groupby(["Year", "Quarter", "Month", "Month_Name", "Year_Month"], as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        gross_profit=("Gross_Profit_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        profit=("Profit_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        avg_discount_pct=("Discount_Pct", "mean"),
    )
)

monthly_kpi["profit_margin"] = monthly_kpi["profit"] / monthly_kpi["net_revenue"].replace(0, np.nan)
monthly_kpi["gross_margin"] = monthly_kpi["gross_profit"] / monthly_kpi["net_revenue"].replace(0, np.nan)
monthly_kpi["contribution_margin"] = monthly_kpi["contribution_profit"] / monthly_kpi["net_revenue"].replace(0, np.nan)
monthly_kpi["cogs_ratio"] = monthly_kpi["cogs"] / monthly_kpi["net_revenue"].replace(0, np.nan)
monthly_kpi["logistics_ratio"] = monthly_kpi["logistics_cost"] / monthly_kpi["net_revenue"].replace(0, np.nan)
monthly_kpi["marketing_ratio"] = monthly_kpi["marketing_spend"] / monthly_kpi["net_revenue"].replace(0, np.nan)
monthly_kpi["avg_selling_price"] = monthly_kpi["net_revenue"] / monthly_kpi["units_sold"].replace(0, np.nan)

monthly_kpi = monthly_kpi.sort_values(["Year", "Month"]).reset_index(drop=True)

monthly_kpi.head()

,Year,Quarter,Month,Month_Name,Year_Month,order_count,units_sold,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,profit,contribution_profit,avg_discount_pct,profit_margin,gross_margin,contribution_margin,cogs_ratio,logistics_ratio,marketing_ratio,avg_selling_price
0,2023,Q1,1,January,2023-01,535,110629,"481,511.88","408,301.70","242,713.37","165,588.33","27,940.41","43,286.62","94,361.30","94,361.30",12.36,0.23,0.41,0.23,0.59,0.07,0.11,3.69
1,2023,Q1,2,February,2023-02,472,98261,"419,997.02","354,938.26","210,194.58","144,743.68","24,395.52","39,437.88","80,910.28","80,910.28",13.14,0.23,0.41,0.23,0.59,0.07,0.11,3.61
2,2023,Q1,3,March,2023-03,512,105895,"438,332.68","371,116.33","222,070.30","149,046.03","24,777.09","40,046.44","84,222.50","84,222.50",12.98,0.23,0.40,0.23,0.60,0.07,0.11,3.50
3,2023,Q2,4,April,2023-04,497,100548,"421,101.02","357,347.48","212,010.72","145,336.76","25,161.47","38,509.72","81,665.57","81,665.57",12.67,0.23,0.41,0.23,0.59,0.07,0.11,3.55
4,2023,Q2,5,May,2023-05,479,99014,"427,223.81","360,818.12","216,709.26","144,108.86","24,584.69","38,239.59","81,284.58","81,284.58",12.96,0.23,0.40,0.23,0.60,0.07,0.11,3.64


In [12]:
# Category KPI table
category_kpi = (
    df.groupby("Product_Category", as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        gross_profit=("Gross_Profit_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        profit=("Profit_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        avg_discount_pct=("Discount_Pct", "mean"),
    )
)

category_kpi["revenue_share"] = category_kpi["net_revenue"] / category_kpi["net_revenue"].sum()
category_kpi["profit_share"] = category_kpi["profit"] / category_kpi["profit"].sum()
category_kpi["profit_margin"] = category_kpi["profit"] / category_kpi["net_revenue"].replace(0, np.nan)
category_kpi["gross_margin"] = category_kpi["gross_profit"] / category_kpi["net_revenue"].replace(0, np.nan)
category_kpi["contribution_margin"] = category_kpi["contribution_profit"] / category_kpi["net_revenue"].replace(0, np.nan)
category_kpi["marketing_ratio"] = category_kpi["marketing_spend"] / category_kpi["net_revenue"].replace(0, np.nan)
category_kpi["marketing_efficiency"] = category_kpi["profit"] / category_kpi["marketing_spend"].replace(0, np.nan)

category_kpi = category_kpi.sort_values("net_revenue", ascending=False).reset_index(drop=True)

category_kpi

,Product_Category,order_count,units_sold,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,profit,contribution_profit,avg_discount_pct,revenue_share,profit_share,profit_margin,gross_margin,contribution_margin,marketing_ratio,marketing_efficiency
0,Beverages,4356,981268,"4,419,617.21","3,737,688.56","2,505,214.86","1,232,473.70","276,179.84","432,092.21","524,201.65","524,201.65",13.02,0.26,0.16,0.14,0.33,0.14,0.12,1.21
1,Household,3237,683008,"3,739,362.54","3,160,720.02","1,775,830.31","1,384,889.71","232,137.42","320,530.71","832,221.58","832,221.58",13.10,0.22,0.25,0.26,0.44,0.26,0.10,2.60
2,Personal Care,3439,686448,"3,266,698.71","2,765,969.05","1,523,818.75","1,242,150.30","176,580.22","288,589.92","776,980.16","776,980.16",12.82,0.19,0.23,0.28,0.45,0.28,0.10,2.69
3,Snacks,4283,912058,"2,973,481.53","2,516,605.10","1,510,797.61","1,005,807.49","158,377.39","292,218.01","555,212.09","555,212.09",12.87,0.17,0.17,0.22,0.40,0.22,0.12,1.90
4,Dairy & Breakfast,2925,620692,"2,681,355.06","2,268,937.27","1,275,821.30","993,115.97","142,264.96","230,501.18","620,349.83","620,349.83",12.85,0.16,0.19,0.27,0.44,0.27,0.10,2.69


In [13]:
# Channel KPI table
channel_kpi = (
    df.groupby("Sales_Channel", as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        profit=("Profit_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        avg_discount_pct=("Discount_Pct", "mean"),
    )
)

channel_kpi["revenue_share"] = channel_kpi["net_revenue"] / channel_kpi["net_revenue"].sum()
channel_kpi["profit_margin"] = channel_kpi["profit"] / channel_kpi["net_revenue"].replace(0, np.nan)
channel_kpi["contribution_margin"] = channel_kpi["contribution_profit"] / channel_kpi["net_revenue"].replace(0, np.nan)
channel_kpi["marketing_ratio"] = channel_kpi["marketing_spend"] / channel_kpi["net_revenue"].replace(0, np.nan)
channel_kpi["marketing_efficiency"] = channel_kpi["profit"] / channel_kpi["marketing_spend"].replace(0, np.nan)

channel_kpi = channel_kpi.sort_values("net_revenue", ascending=False).reset_index(drop=True)

channel_kpi

,Sales_Channel,order_count,units_sold,gross_sales,net_revenue,cogs,logistics_cost,marketing_spend,profit,contribution_profit,avg_discount_pct,revenue_share,profit_margin,contribution_margin,marketing_ratio,marketing_efficiency
0,Wholesale,2848,1456048,"6,363,163.57","5,208,657.92","3,158,966.47","280,998.50","419,413.70","1,349,279.25","1,349,279.25",18.07,0.36,0.26,0.26,0.08,3.22
1,Distributor,3629,1227016,"5,293,893.40","4,451,811.56","2,620,527.07","284,669.62","423,220.89","1,123,393.98","1,123,393.98",15.79,0.31,0.25,0.25,0.10,2.65
2,Modern Trade,6219,860610,"3,866,687.09","3,371,020.34","1,979,798.74","267,043.00","463,912.39","660,266.21","660,266.21",12.55,0.23,0.20,0.20,0.14,1.42
3,Online,5544,339800,"1,556,770.99","1,418,430.18","832,190.55","152,828.71","257,385.05","176,025.87","176,025.87",8.86,0.10,0.12,0.12,0.18,0.68


In [14]:
# Region and country KPI tables

region_kpi = (
    df.groupby("Region", as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
    )
)

region_kpi["revenue_share"] = region_kpi["net_revenue"] / region_kpi["net_revenue"].sum()
region_kpi["profit_margin"] = region_kpi["profit"] / region_kpi["net_revenue"].replace(0, np.nan)
region_kpi["marketing_ratio"] = region_kpi["marketing_spend"] / region_kpi["net_revenue"].replace(0, np.nan)
region_kpi["logistics_ratio"] = region_kpi["logistics_cost"] / region_kpi["net_revenue"].replace(0, np.nan)

region_kpi = region_kpi.sort_values("net_revenue", ascending=False).reset_index(drop=True)


country_kpi = (
    df.groupby(["Region", "Country"], as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
    )
)

country_kpi["revenue_share"] = country_kpi["net_revenue"] / country_kpi["net_revenue"].sum()
country_kpi["profit_margin"] = country_kpi["profit"] / country_kpi["net_revenue"].replace(0, np.nan)
country_kpi["marketing_ratio"] = country_kpi["marketing_spend"] / country_kpi["net_revenue"].replace(0, np.nan)
country_kpi["logistics_ratio"] = country_kpi["logistics_cost"] / country_kpi["net_revenue"].replace(0, np.nan)

country_kpi = country_kpi.sort_values("net_revenue", ascending=False).reset_index(drop=True)

region_kpi.head(), country_kpi.head()

(          Region  order_count  units_sold  net_revenue       profit  \
 0         Europe         5532     1168906 4,811,497.42 1,135,420.40   
 1  North America         4150      876372 3,337,777.37   780,169.10   
 2           Asia         4420      941879 2,946,505.41   670,320.46   
 3  South America         2303      510620 1,728,515.24   404,226.59   
 4        Oceania         1835      385697 1,625,624.56   318,828.76   
 
    marketing_spend  logistics_cost  revenue_share  profit_margin  \
 0       523,676.26      312,738.43           0.33           0.24   
 1       368,430.87      216,987.85           0.23           0.23   
 2       317,712.33      201,231.27           0.20           0.23   
 3       179,731.07      124,377.39           0.12           0.23   
 4       174,381.50      130,204.89           0.11           0.20   
 
    marketing_ratio  logistics_ratio  
 0             0.11             0.06  
 1             0.11             0.07  
 2             0.11             0

In [15]:
# Promotion KPI table
promotion_kpi = (
    df.groupby("Promotion_Type", as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        avg_discount_pct=("Discount_Pct", "mean"),
    )
)

promotion_kpi["revenue_share"] = promotion_kpi["net_revenue"] / promotion_kpi["net_revenue"].sum()
promotion_kpi["profit_margin"] = promotion_kpi["profit"] / promotion_kpi["net_revenue"].replace(0, np.nan)
promotion_kpi["marketing_ratio"] = promotion_kpi["marketing_spend"] / promotion_kpi["net_revenue"].replace(0, np.nan)
promotion_kpi["marketing_efficiency"] = promotion_kpi["profit"] / promotion_kpi["marketing_spend"].replace(0, np.nan)

promotion_kpi = promotion_kpi.sort_values("net_revenue", ascending=False).reset_index(drop=True)

promotion_kpi

,Promotion_Type,order_count,units_sold,net_revenue,profit,marketing_spend,avg_discount_pct,revenue_share,profit_margin,marketing_ratio,marketing_efficiency
0,No Promo,6741,1438634,"5,587,181.47","1,432,411.66","452,421.75",8.86,0.39,0.26,0.08,3.17
1,Bundle Offer,2784,615463,"2,220,908.58","487,231.45","259,855.95",14.82,0.15,0.22,0.12,1.88
2,Seasonal Campaign,2895,602478,"2,216,237.58","477,538.93","266,508.37",13.82,0.15,0.22,0.12,1.79
3,Flash Discount,2140,452027,"1,593,444.48","306,545.88","236,780.34",17.72,0.11,0.19,0.15,1.29
4,Festival Campaign,1531,329545,"1,183,731.71","218,696.37","178,390.14",16.55,0.08,0.18,0.15,1.23
5,Loyalty Cashback,1398,293784,"1,114,225.30","281,314.87","91,062.17",12.23,0.08,0.25,0.08,3.09
6,Introductory Offer,751,151543,"534,190.88","105,226.15","78,913.31",19.46,0.04,0.20,0.15,1.33


In [16]:
# Product/SKU KPI table
product_kpi = (
    df.groupby(["Product_Category", "Brand", "Product_Name", "SKU"], as_index=False)
    .agg(
        order_count=("Order_ID", "nunique"),
        units_sold=("Units_Sold", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        avg_discount_pct=("Discount_Pct", "mean"),
        avg_unit_price=("Unit_Price_USD", "mean"),
    )
)

product_kpi["revenue_share"] = product_kpi["net_revenue"] / product_kpi["net_revenue"].sum()
product_kpi["profit_share"] = product_kpi["profit"] / product_kpi["profit"].sum()
product_kpi["profit_margin"] = product_kpi["profit"] / product_kpi["net_revenue"].replace(0, np.nan)
product_kpi["marketing_ratio"] = product_kpi["marketing_spend"] / product_kpi["net_revenue"].replace(0, np.nan)
product_kpi["marketing_efficiency"] = product_kpi["profit"] / product_kpi["marketing_spend"].replace(0, np.nan)
product_kpi["avg_selling_price"] = product_kpi["net_revenue"] / product_kpi["units_sold"].replace(0, np.nan)

product_kpi = product_kpi.sort_values("net_revenue", ascending=False).reset_index(drop=True)

product_kpi.head(10)

,Product_Category,Brand,Product_Name,SKU,order_count,units_sold,net_revenue,profit,marketing_spend,cogs,logistics_cost,avg_discount_pct,avg_unit_price,revenue_share,profit_share,profit_margin,marketing_ratio,marketing_efficiency,avg_selling_price
0,Beverages,RoastTrail,Instant Coffee Gold,BEV-RT-301,653,159112,"1,169,394.83","114,301.14","135,683.12","833,235.73","86,174.84",13.22,8.88,0.08,0.03,0.10,0.12,0.84,7.35
1,Household,HomeNest,Laundry Detergent,HHD-HN-102,520,113682,"927,378.97","239,943.56","92,526.51","528,097.80","66,811.10",13.10,9.81,0.06,0.07,0.26,0.10,2.59,8.16
2,Personal Care,PureLiva,Shampoo Repair,PCR-PL-101,594,122128,"690,564.13","178,437.27","70,636.98","397,642.38","43,847.50",12.83,6.82,0.05,0.05,0.26,0.10,2.53,5.65
3,Beverages,FuelCore,Energy Drink Zero,BEV-FC-202,851,198116,"673,154.48","138,004.99","75,722.32","409,840.96","49,586.21",13.03,4.11,0.05,0.04,0.21,0.11,1.82,3.40
4,Personal Care,PureLiva,Conditioner Smooth,PCR-PL-102,541,105615,"631,746.18","150,884.60","69,346.87","370,882.64","40,632.07",12.85,7.22,0.04,0.05,0.24,0.11,2.18,5.98
5,Snacks,TrailNest,Trail Mix Original,SNK-TN-301,699,152606,"608,390.24","136,024.65","67,382.08","366,870.56","38,112.95",12.81,4.84,0.04,0.04,0.22,0.11,2.02,3.99
6,Beverages,ZenLeaf,Green Tea Bags,BEV-ZL-401,656,138110,"583,283.43","83,331.85","67,297.99","389,325.69","43,327.90",12.82,5.10,0.04,0.03,0.14,0.12,1.24,4.22
7,Beverages,FuelCore,Energy Drink Classic,BEV-FC-201,823,180188,"582,851.49","120,061.86","66,971.37","352,696.92","43,121.34",13.00,3.90,0.04,0.04,0.21,0.11,1.79,3.23
8,Dairy & Breakfast,MorningCo,Granola Honey,DBF-MC-101,503,107922,"561,235.32","143,632.62","59,656.02","322,354.63","35,592.05",12.79,6.25,0.04,0.04,0.26,0.11,2.41,5.20
9,Personal Care,FreshNest,Body Wash Citrus,PCR-FN-201,591,113411,"547,875.08","145,164.99","55,346.28","311,929.51","35,434.30",12.71,5.82,0.04,0.04,0.26,0.10,2.62,4.83


In [17]:
# Executive summary table

executive_summary = pd.DataFrame({
    "metric": [
        "Total Orders",
        "Total Units Sold",
        "Gross Sales",
        "Net Revenue",
        "COGS",
        "Gross Profit",
        "Logistics Cost",
        "Marketing Spend",
        "Profit",
        "Contribution Profit",
        "Profit Margin",
        "Gross Margin",
        "Contribution Margin",
        "COGS Ratio",
        "Logistics Cost Ratio",
        "Marketing Spend Ratio",
    ],
    "value": [
        df["Order_ID"].nunique(),
        df["Units_Sold"].sum(),
        df["Gross_Sales_USD"].sum(),
        df["Net_Revenue_USD"].sum(),
        df["COGS_USD"].sum(),
        df["Gross_Profit_USD"].sum(),
        df["Logistics_Cost_USD"].sum(),
        df["Marketing_Spend_USD"].sum(),
        df["Profit_USD"].sum(),
        df["Contribution_Profit_USD"].sum(),
        df["Profit_USD"].sum() / df["Net_Revenue_USD"].sum(),
        df["Gross_Profit_USD"].sum() / df["Net_Revenue_USD"].sum(),
        df["Contribution_Profit_USD"].sum() / df["Net_Revenue_USD"].sum(),
        df["COGS_USD"].sum() / df["Net_Revenue_USD"].sum(),
        df["Logistics_Cost_USD"].sum() / df["Net_Revenue_USD"].sum(),
        df["Marketing_Spend_USD"].sum() / df["Net_Revenue_USD"].sum(),
    ]
})

executive_summary

,metric,value
0,Total Orders,"18,240.00"
1,Total Units Sold,"3,883,474.00"
2,Gross Sales,"17,080,515.05"
3,Net Revenue,"14,449,920.00"
4,COGS,"8,591,482.83"
5,Gross Profit,"5,858,437.17"
6,Logistics Cost,"985,539.83"
7,Marketing Spend,"1,563,932.03"
8,Profit,"3,308,965.31"
9,Contribution Profit,"3,308,965.31"


# 7. Save Processed Artifacts

In [18]:
df.to_csv(PROCESSED_DATA_DIR / "fmcg_cleaned.csv", index=False)
monthly_kpi.to_csv(PROCESSED_DATA_DIR / "monthly_kpi.csv", index=False)
category_kpi.to_csv(PROCESSED_DATA_DIR / "category_kpi.csv", index=False)
channel_kpi.to_csv(PROCESSED_DATA_DIR / "channel_kpi.csv", index=False)
region_kpi.to_csv(PROCESSED_DATA_DIR / "region_kpi.csv", index=False)
country_kpi.to_csv(PROCESSED_DATA_DIR / "country_kpi.csv", index=False)
promotion_kpi.to_csv(PROCESSED_DATA_DIR / "promotion_kpi.csv", index=False)
product_kpi.to_csv(PROCESSED_DATA_DIR / "product_kpi.csv", index=False)
executive_summary.to_csv(PROCESSED_DATA_DIR / "executive_summary.csv", index=False)

print("Saved processed artifacts to:", PROCESSED_DATA_DIR)

Saved processed artifacts to: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed


In [19]:
saved_files = [
    "fmcg_cleaned.csv",
    "monthly_kpi.csv",
    "category_kpi.csv",
    "channel_kpi.csv",
    "region_kpi.csv",
    "country_kpi.csv",
    "promotion_kpi.csv",
    "product_kpi.csv",
    "executive_summary.csv",
]

for file in saved_files:
    path = PROCESSED_DATA_DIR / file
    temp = pd.read_csv(path)
    print(f"{file}: {temp.shape[0]:,} rows × {temp.shape[1]:,} columns")

fmcg_cleaned.csv: 18,240 rows × 52 columns
monthly_kpi.csv: 36 rows × 23 columns
category_kpi.csv: 5 rows × 19 columns
channel_kpi.csv: 4 rows × 16 columns
region_kpi.csv: 5 rows × 11 columns
country_kpi.csv: 17 rows × 12 columns
promotion_kpi.csv: 7 rows × 11 columns
product_kpi.csv: 30 rows × 19 columns
executive_summary.csv: 16 rows × 2 columns
